In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Fig. 4.1 (Curtain plot) generator — axial/coronal/sagittal candidates

- Reuses your existing loader + constants from multimodal_qc_tasks_extra.py
- Builds a slice-wise curtain: 351 modalities × ROI pixels (in ONE 2D slice)
- For each axis, selects the slice with maximum ROI-mask coverage
- Saves: PDF (for LaTeX) + PNG (600 dpi) for each axis
- Writes a meta JSON for reproducibility
"""

from __future__ import annotations
import sys
import json
from pathlib import Path
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# Ensure local imports work when running as a script
THIS_DIR = Path(__file__).resolve().parent
sys.path.insert(0, str(THIS_DIR))

import multimodal_qc_tasks_extra as qc  # uses your loader + constants  :contentReference[oaicite:1]{index=1}


# -------------------------
# Thesis-style matplotlib defaults (safe on Linux)
# -------------------------
def set_thesis_style():
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans", "Arial", "Liberation Sans"],
        "font.size": 9,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "axes.linewidth": 0.6,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        "xtick.major.size": 3.0,
        "ytick.major.size": 3.0,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "axes.unicode_minus": False,
        # Embed TrueType fonts in PDF (good for thesis + editing)
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.facecolor": "white",
        "savefig.facecolor": "white",
    })


# -------------------------
# Slice selection: maximize ROI-mask coverage
# -------------------------
def choose_slice_max_mask(mask3d: np.ndarray, axis: int, min_ratio: float = 0.05) -> tuple[int, float]:
    m = mask3d.astype(bool)
    if axis == 0:   # sagittal
        ratios = m.reshape(m.shape[0], -1).mean(axis=1)
    elif axis == 1: # coronal
        ratios = m.transpose(1, 0, 2).reshape(m.shape[1], -1).mean(axis=1)
    elif axis == 2: # axial
        ratios = m.transpose(2, 0, 1).reshape(m.shape[2], -1).mean(axis=1)
    else:
        raise ValueError(f"Invalid axis: {axis}")

    valid = np.where(ratios >= min_ratio)[0]
    if valid.size == 0:
        k = int(np.argmax(ratios))
        return k, float(ratios[k])

    k = int(valid[np.argmax(ratios[valid])])
    return k, float(ratios[k])


def extract_slice_hw_c(data_4d: np.ndarray, axis: int, slice_idx: int) -> np.ndarray:
    # data_4d: (X, Y, Z, C)
    if axis == 0:   # sagittal -> (Y, Z, C)
        return data_4d[slice_idx, :, :, :]
    if axis == 1:   # coronal  -> (X, Z, C)
        return data_4d[:, slice_idx, :, :]
    # axis == 2 axial -> (X, Y, C)
    return data_4d[:, :, slice_idx, :]


# -------------------------
# Curtain computation (slice-wise)
# -------------------------
def compute_curtain_for_slice(
    data_4d: np.ndarray,
    mask3d: np.ndarray,
    axis: int,
    slice_idx: int,
    ref_modality_idx: int = 341,
    max_pixels: int = 50_000,
    sort_by: str | None = "mprage",
    random_seed: int = 42,
) -> tuple[np.ndarray, dict]:
    """
    Returns:
      curtain_u8: uint8 array, shape = (351, N_pixels_used)
      meta: dict with parameters and selected slice stats
    """
    slice_hw_c = extract_slice_hw_c(data_4d, axis, slice_idx)  # (H,W,C)
    mask2d = qc.get_slice_2d(mask3d.astype(bool), axis, slice_idx).astype(bool)

    H, W, C = slice_hw_c.shape
    if C != 351:
        raise ValueError(f"Expected 351 channels, got {C}")

    idx_all = np.flatnonzero(mask2d.ravel())
    if idx_all.size == 0:
        raise ValueError("Selected slice has empty ROI mask; cannot build curtain.")

    rng = np.random.default_rng(random_seed)
    idx = idx_all
    if idx_all.size > max_pixels:
        idx = rng.choice(idx_all, size=max_pixels, replace=False)

    flat = slice_hw_c.reshape(-1, C).astype(np.float32)  # (H*W, C)
    V = flat[idx, :]  # (N, C)

    if sort_by == "mprage":
        order = np.argsort(V[:, ref_modality_idx])
        V = V[order, :]

    curtain_u8 = np.empty((C, V.shape[0]), dtype=np.uint8)
    for ch in range(C):
        v = V[:, ch]
        p1, p99 = np.percentile(v, [1, 99])
        vn = np.clip((v - p1) / (p99 - p1 + 1e-8), 0.0, 1.0)
        curtain_u8[ch, :] = (vn * 255.0).astype(np.uint8)

    meta = {
        "axis": int(axis),
        "slice_idx": int(slice_idx),
        "mask_ratio": float(mask2d.mean()),
        "n_pixels_total": int(idx_all.size),
        "n_pixels_used": int(V.shape[0]),
        "sort_by": sort_by,
        "ref_modality_idx": int(ref_modality_idx),
        "random_seed": int(random_seed),
        "max_pixels": int(max_pixels),
        "normalization": "per-modality percentile (1,99) within selected pixels",
    }
    return curtain_u8, meta


# -------------------------
# Plot & save (thesis-friendly: PDF + 600dpi PNG)
# -------------------------
def plot_and_save_curtain(
    curtain_u8: np.ndarray,
    meta: dict,
    out_base: Path,
    axis_name: str,
    add_group_lines: bool = True,
    dpi_png: int = 600,
):
    fig, ax = plt.subplots(figsize=(7.0, 4.2))  # ~1-column to page-width friendly
    im = ax.imshow(
        curtain_u8,
        cmap="gray",
        aspect="auto",
        interpolation="nearest",
        rasterized=True,  # keep PDF size reasonable
    )

    # Keep figure text minimal; details should go into LaTeX caption
    if meta.get("sort_by") == "mprage":
        ax.set_xlabel(f"ROI pixels (sorted by MPRAGE; n={meta['n_pixels_used']:,})")
    else:
        ax.set_xlabel(f"ROI pixels (sampled; n={meta['n_pixels_used']:,})")
    ax.set_ylabel("Modality index (1–351)")

    # Remove dense ticks (not readable at this resolution)
    ax.set_xticks([])
    ax.set_yticks([])

    if add_group_lines:
        # Optional: light horizontal separators for modality families/sub-families
        for _, a, b in qc.GROUPS_FOR_BOUNDS:
            ax.axhline(b + 0.5, lw=0.4, color="0.75")

    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Normalized intensity (1–99% per modality)")

    fig.tight_layout()
    out_base.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight", pad_inches=0.01)
    fig.savefig(out_base.with_suffix(".png"), dpi=dpi_png, bbox_inches="tight", pad_inches=0.01)
    plt.close(fig)


def main():
    set_thesis_style()

    data_dir = Path(qc.DATA_DIR)  # comes from your existing scripts 
    mat_files = sorted(data_dir.glob("*.mat"))
    if not mat_files:
        raise FileNotFoundError(f"No .mat files found in: {data_dir}")

    # ---- choose ONE subject (edit here if you want a specific patient)
    patient_path = mat_files[0]
    subject_id = patient_path.stem

    d = qc.load_minimal_3d_data(patient_path)  # exact same loader 
    data_4d = d["data"]
    mask3d = d["region_mask"].astype(bool)

    ref_idx = int(getattr(qc, "REFERENCE_MODALITY", 341))

    # ---- output folder (edit if you want)
    out_dir = Path(qc.BASE_OUTPUT_DIR) / "thesis_figures" / "chapter4" / "fig4_1_curtain"
    out_dir.mkdir(parents=True, exist_ok=True)

    axes = [("sagittal", 0), ("coronal", 1), ("axial", 2)]
    all_meta = {"subject": subject_id, "source_mat": str(patient_path), "outputs": []}

    for axis_name, axis in axes:
        slice_idx, _ = choose_slice_max_mask(mask3d, axis=axis, min_ratio=0.05)

        curtain_u8, meta = compute_curtain_for_slice(
            data_4d=data_4d,
            mask3d=mask3d,
            axis=axis,
            slice_idx=slice_idx,
            ref_modality_idx=ref_idx,
            max_pixels=50_000,
            sort_by="mprage",
            random_seed=42,
        )

        out_base = out_dir / f"fig4_1_curtain_{subject_id}_{axis_name}_slice{slice_idx:03d}"
        plot_and_save_curtain(
            curtain_u8=curtain_u8,
            meta=meta,
            out_base=out_base,
            axis_name=axis_name,
            add_group_lines=True,
            dpi_png=600,
        )

        meta["axis_name"] = axis_name
        meta["out_pdf"] = str(out_base.with_suffix(".pdf"))
        meta["out_png"] = str(out_base.with_suffix(".png"))
        all_meta["outputs"].append(meta)

    meta_path = out_dir / f"fig4_1_curtain_{subject_id}_meta.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(all_meta, f, indent=2, ensure_ascii=False)

    print(f"[OK] Fig 4.1 candidates saved to: {out_dir}")
    print(f"[OK] Meta written to: {meta_path}")


if __name__ == "__main__":
    main()
